# Signal Synthesis with wavesst

This notebook demonstrates the `wavesst.synthesis` module, which provides three signal generators:

- `make_chirp` — chirp signals with linear, quadratic, or arbitrary instantaneous frequency
- `make_amfm` — AM/FM modulated carrier signals
- `make_noise` — coloured noise (white, pink, brown, impulsive)

All generators return `float32` NumPy arrays of shape `(N,)` where `N = int(duration * fs)`.

Topics:
1. Linear and quadratic chirps
2. Arbitrary instantaneous frequency (sinusoidal IF)
3. Gated signals — onset/offset windowing
4. Noise synthesis — waveforms and power spectra
5. Signal + noise — CWT and SST ridge comparison

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import wavesst

%matplotlib inline

cfg = wavesst.Config(device='cpu', dtype='complex128')
FS = 256.0   # sample rate used throughout
print(f'wavesst {wavesst.__version__}')

## 1 — Linear and Quadratic Chirps

A **linear chirp** sweeps frequency at a constant rate:

$$f(t) = f_{\text{start}} + (f_{\text{end}} - f_{\text{start}}) \frac{t}{T}$$

A **quadratic chirp** accelerates the sweep:

$$f(t) = f_{\text{start}} + (f_{\text{end}} - f_{\text{start}}) \left(\frac{t}{T}\right)^2$$

Both sweep 20 → 80 Hz over 2 seconds at fs = 256 Hz.  
The SST concentrates the broadened CWT energy onto the true IF ridge.

In [ ]:
DURATION = 2.0
t = np.arange(int(DURATION * FS)) / FS

chirp_lin = wavesst.make_chirp(DURATION, FS, f_start=20, f_end=80, method='linear')
chirp_quad = wavesst.make_chirp(DURATION, FS, f_start=20, f_end=80, method='quadratic')

fig, axes = plt.subplots(2, 1, figsize=(12, 4), sharex=True)
axes[0].plot(t, chirp_lin, lw=0.7, color='steelblue')
axes[0].set_title('Linear chirp  (20 → 80 Hz)')
axes[0].set_ylabel('Amplitude')
axes[1].plot(t, chirp_quad, lw=0.7, color='darkorange')
axes[1].set_title('Quadratic chirp  (20 → 80 Hz)')
axes[1].set_ylabel('Amplitude')
axes[1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

In [ ]:
sst_lin  = wavesst.sst(chirp_lin,  fs=FS, nv=32, gamma='auto', cfg=cfg)
sst_quad = wavesst.sst(chirp_quad, fs=FS, nv=32, gamma='auto', cfg=cfg)

# True IF curves for overlay
if_linear = 20 + 60 * (t / DURATION)
if_quad   = 20 + 60 * (t / DURATION) ** 2

fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharey=True)

for row, (result, if_true, chirp_name) in enumerate(zip(
    [sst_lin, sst_quad],
    [if_linear, if_quad],
    ['Linear chirp', 'Quadratic chirp'],
)):
    Tx_mag = result.Tx.abs().numpy()
    ax_sst, ax_if = axes[row]

    for ax in (ax_sst, ax_if):
        ax.pcolormesh(result.times, result.freqs, Tx_mag,
                      shading='auto', cmap='inferno')
        ax.set_ylim(0, 128)
        ax.set_xlabel('Time (s)')

    ax_sst.set_ylabel('Frequency (Hz)')
    ax_sst.set_title(f'SST — {chirp_name}')
    ax_if.set_title(f'SST + true IF — {chirp_name}')
    ax_if.plot(t, if_true, 'w--', lw=1.2, label='True IF')
    ax_if.legend(loc='upper left')

plt.tight_layout()
plt.show()

## 2 — Arbitrary Instantaneous Frequency

By passing `method='arbitrary'` and a callable `f_inst`, the chirp generator integrates any IF trajectory:

$$\phi(t) = 2\pi \int_0^t f_{\text{inst}}(\tau)\, d\tau$$

Here we use a **sinusoidal IF**:

$$f_{\text{inst}}(t) = 50 + 30\sin(2\pi \cdot 0.5 \cdot t)$$

This sweeps between 20 Hz and 80 Hz following a sinusoidal pattern.  
The SST ridge (extracted by dynamic programming) should track the true IF curve closely.

In [ ]:
f_inst = lambda t_arr: 50 + 30 * np.sin(2 * np.pi * 0.5 * t_arr)

chirp_arb = wavesst.make_chirp(DURATION, FS, method='arbitrary', f_inst=f_inst)

# Time domain waveform
fig, ax = plt.subplots(figsize=(12, 2.5))
ax.plot(t, chirp_arb, lw=0.6, color='teal')
ax.set_title('Arbitrary IF chirp — sinusoidal instantaneous frequency (50 ± 30 Hz)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
plt.tight_layout()
plt.show()

In [ ]:
sst_arb = wavesst.sst(chirp_arb, fs=FS, nv=32, gamma='auto', cfg=cfg)

# Extract the dominant ridge
ridges_arb = wavesst.extract_ridges(sst_arb, n=1, penalty=2.0)
ridge_arb  = ridges_arb[0]

# True IF evaluated at sample times
if_true_arb = f_inst(t)

Tx_mag = sst_arb.Tx.abs().numpy()
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(18, 4))

# Left: SST only
ax1.pcolormesh(sst_arb.times, sst_arb.freqs, Tx_mag,
               shading='auto', cmap='inferno')
ax1.set_ylim(0, 128)
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Frequency (Hz)')
ax1.set_title('SST — Arbitrary (sinusoidal) IF chirp')

# Middle: SST + true IF overlay
ax2.pcolormesh(sst_arb.times, sst_arb.freqs, Tx_mag,
               shading='auto', cmap='inferno')
ax2.plot(t, if_true_arb, 'w--', lw=1.5, label='True IF')
ax2.set_ylim(0, 128)
ax2.set_xlabel('Time (s)')
ax2.set_title('SST + true IF')
ax2.legend(loc='upper right', fontsize=8)

# Right: extracted ridge vs true IF
ax3.plot(ridge_arb.times, if_true_arb, 'k--', lw=1.5, label='True IF')
ax3.plot(ridge_arb.times, ridge_arb.freq_path, 'r', lw=1.2, label='SST ridge')
ax3.set_xlabel('Time (s)')
ax3.set_ylabel('Frequency (Hz)')
ax3.set_title('Ridge tracking — true IF vs extracted')
ax3.legend()
ax3.set_ylim(0, 100)

plt.tight_layout()
plt.show()

error_hz = np.abs(ridge_arb.freq_path - if_true_arb)
print(f'Ridge tracking error: mean={error_hz.mean():.2f} Hz,  max={error_hz.max():.2f} Hz')

## 3 — Gated Signals (Onset/Offset Windowing)

`make_chirp` (and `make_amfm`) accept `t_start` and `t_end` parameters that gate the signal:
samples outside `[t_start, t_end]` are set to zero.  This models physical phenomena that
are active only during a sub-interval — a bird call, a mechanical transient, a musical note.

Here we create a **40 Hz tone** active from **t = 0.5 s to t = 1.5 s** inside a 2-second window.
`detect_onsets` estimates the active interval from the ridge energy profile.

In [ ]:
tone_gated = wavesst.make_chirp(
    DURATION, FS,
    f_start=40, f_end=40,   # constant 40 Hz
    method='linear',
    t_start=0.5, t_end=1.5,
)

fig, ax = plt.subplots(figsize=(12, 2.5))
ax.plot(t, tone_gated, lw=0.7, color='purple')
ax.axvspan(0.5, 1.5, color='gold', alpha=0.25, label='Active window')
ax.set_title('Gated 40 Hz tone  (active 0.5 s – 1.5 s)')
ax.set_xlabel('Time (s)')
ax.set_ylabel('Amplitude')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
sst_gated = wavesst.sst(tone_gated, fs=FS, nv=32, gamma='auto', cfg=cfg)

ridges_gated = wavesst.extract_ridges(sst_gated, n=1, penalty=1.0)
ridge_gated  = ridges_gated[0]

onset_result = wavesst.detect_onsets(ridge_gated)
print(f'Detected onset : {onset_result.t_start:.3f} s  (true: 0.500 s)')
print(f'Detected offset: {onset_result.t_end:.3f} s  (true: 1.500 s)')

Tx_mag = sst_gated.Tx.abs().numpy()
fig, ax = plt.subplots(figsize=(12, 4))
ax.pcolormesh(sst_gated.times, sst_gated.freqs, Tx_mag,
              shading='auto', cmap='inferno')
ax.axvline(onset_result.t_start, color='cyan', lw=2, linestyle='--',
           label=f'Onset  {onset_result.t_start:.3f} s')
ax.axvline(onset_result.t_end, color='lime', lw=2, linestyle='--',
           label=f'Offset {onset_result.t_end:.3f} s')
ax.set_ylim(0, 80)
ax.set_xlabel('Time (s)')
ax.set_ylabel('Frequency (Hz)')
ax.set_title('SST — gated 40 Hz tone with detected onset/offset markers')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 4 — Noise Synthesis

`make_noise` supports four noise colours:

| Colour | Spectral density | Notes |
|--------|-----------------|-------|
| **white** | flat | equal power per Hz |
| **pink** | 1/√f | equal power per octave |
| **brown** | 1/f | random walk / Brownian motion |
| **impulsive** | broadband | sparse random spikes (~1 % non-zero) |

Pink and brown noise are generated by shaping white noise in the frequency domain then inverting.

In [ ]:
colors = ['white', 'pink', 'brown', 'impulsive']
noises = {c: wavesst.make_noise(DURATION, FS, color=c, seed=42) for c in colors}

fig, axes = plt.subplots(2, 2, figsize=(14, 5), sharex=True)
palette = ['#4878cf', '#e87b2b', '#6acc65', '#d65f5f']

for ax, (color, x), col in zip(axes.flat, noises.items(), palette):
    ax.plot(t, x, lw=0.5, color=col)
    ax.set_title(f'{color.capitalize()} noise')
    ax.set_ylabel('Amplitude')

for ax in axes[1]:
    ax.set_xlabel('Time (s)')

plt.suptitle('Noise waveforms — time domain', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Power spectral density — log-log scale to see 1/f slope
fig, axes = plt.subplots(2, 2, figsize=(14, 5))

expected_slopes = {
    'white': (0.0, 'flat'),
    'pink': (-1.0, '–10 dB/decade'),
    'brown': (-2.0, '–20 dB/decade'),
    'impulsive': (0.0, 'broadband'),
}

for ax, (color, x), col in zip(axes.flat, noises.items(), palette):
    N = len(x)
    freqs_fft = np.fft.rfftfreq(N, d=1.0 / FS)
    psd = np.abs(np.fft.rfft(x)) ** 2 / N

    # Avoid log(0)
    mask = freqs_fft > 0
    ax.loglog(freqs_fft[mask], psd[mask], color=col, lw=0.8, alpha=0.85)

    slope, label = expected_slopes[color]
    ax.set_title(f'{color.capitalize()} noise  ({label})')
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PSD')

plt.suptitle('Power spectral density — log-log scale', fontsize=13)
plt.tight_layout()
plt.show()

## 5 — Signal + Noise

Real signals are always observed with noise.  Here we add **pink noise** (amplitude = 0.3)
to a linear chirp and compare:

1. **CWT** — how much the broadened ridge survives in noise
2. **SST ridge** — the concentrated ridge path is more robust than raw CWT energy

Pink noise is a natural choice here: it has more low-frequency energy than white noise
and mimics many real-world backgrounds (electronics, wind, biological signals).

In [ ]:
chirp_clean = wavesst.make_chirp(DURATION, FS, f_start=30, f_end=100, method='linear')
noise_pink  = wavesst.make_noise(DURATION, FS, color='pink', amplitude=0.3, seed=7)
chirp_noisy = (chirp_clean + noise_pink).astype(np.float32)

# Quick SNR estimate
snr_db = 20 * np.log10(np.sqrt(np.mean(chirp_clean**2)) / np.sqrt(np.mean(noise_pink**2)))
print(f'Signal RMS: {np.sqrt(np.mean(chirp_clean**2)):.3f}')
print(f'Noise  RMS: {np.sqrt(np.mean(noise_pink**2)):.3f}   (amplitude=0.3)')
print(f'SNR:        {snr_db:.1f} dB')

In [ ]:
cwt_clean = wavesst.cwt(chirp_clean, fs=FS, nv=32, cfg=cfg)
cwt_noisy = wavesst.cwt(chirp_noisy, fs=FS, nv=32, cfg=cfg)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, result, title in zip(
    axes,
    [cwt_clean, cwt_noisy],
    ['CWT — clean chirp (30 → 100 Hz)', 'CWT — chirp + pink noise (SNR ≈ {:.0f} dB)'.format(snr_db)],
):
    W_mag = result.W.abs().numpy()
    ax.pcolormesh(result.times, result.freqs, W_mag,
                  shading='auto', cmap='magma')
    ax.set_ylim(0, 128)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title(title)

plt.tight_layout()
plt.show()

In [ ]:
sst_clean = wavesst.sst(chirp_clean, fs=FS, nv=32, gamma='auto', cfg=cfg)
sst_noisy = wavesst.sst(chirp_noisy, fs=FS, nv=32, gamma='auto', cfg=cfg)

ridge_clean = wavesst.extract_ridges(sst_clean, n=1, penalty=2.0)[0]
ridge_noisy = wavesst.extract_ridges(sst_noisy, n=1, penalty=2.0)[0]

if_true_5 = 30 + 70 * (t / DURATION)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, result, ridge, title in zip(
    axes,
    [sst_clean, sst_noisy],
    [ridge_clean, ridge_noisy],
    ['SST — clean chirp', 'SST — chirp + pink noise'],
):
    Tx_mag = result.Tx.abs().numpy()
    ax.pcolormesh(result.times, result.freqs, Tx_mag,
                  shading='auto', cmap='magma')
    ax.plot(t, if_true_5, 'w--', lw=1.5, label='True IF', alpha=0.8)
    ax.plot(ridge.times, ridge.freq_path, 'cyan', lw=1.2, label='Extracted ridge')
    ax.set_ylim(0, 128)
    ax.set_xlabel('Time (s)')
    ax.set_ylabel('Frequency (Hz)')
    ax.set_title(title)
    ax.legend(loc='upper left')

plt.tight_layout()
plt.show()

err_clean = np.abs(ridge_clean.freq_path - if_true_5)
err_noisy = np.abs(ridge_noisy.freq_path - if_true_5)
print(f'Clean ridge error: mean={err_clean.mean():.2f} Hz,  max={err_clean.max():.2f} Hz')
print(f'Noisy ridge error: mean={err_noisy.mean():.2f} Hz,  max={err_noisy.max():.2f} Hz')